In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")
api_key = os.getenv("DART_API_KEY")
print(f"API 키 로드: {'✅' if api_key else '❌'}")

API 키 로드: ✅


In [ ]:
# 삼성전자 2024년 사업보고서 (연결재무제표) 호출
# corp_code: 00126380 (Day 2에서 매핑한 값)
# bsns_year: 2024 (사업연도)
#  reprt_code: 11011 (사업보고서 = 4분기, 연간)
# 💡 reprt_code 코드표:
# 11011: 사업보고서 (4분기, 연간)
# 11014: 3분기보고서
# 11012: 반기보고서 (2분기)
# 11013: 1분기보고서
# fs_div: CFS (연결재무제표)

url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
params = {
    "crtfc_key": api_key,
    "corp_code": "00126380",
    "bsns_year": "2026",
    "reprt_code": "11013",  # 사업보고서
    "fs_div": "CFS",        # 연결재무제표
}

response = requests.get(url, params=params)
data = response.json()

print(f"응답 상태: {data.get('status')}")
print(f"메시지: {data.get('message')}")
print(f"항목 수: {len(data.get('list', []))}")

응답 상태: 000
메시지: 정상
항목 수: 215


In [ ]:
# list 부분을 DataFrame으로 변환
# 💡 핵심 컬럼 의미:
# sj_nm: 재무제표 종류 (손익계산서/재무상태표/현금흐름표)
# account_id: K-IFRS 표준 계정 코드 (영문)
# account_nm: 계정 이름 (한글)
# thstrm_amount: 당기금액 (이번 기간)
# frmtrm_amount: 전기금액 (전년 동기)
# df = pd.DataFrame(data["list"])

print(f"전체 행 수: {len(df)}")
print(f"\n=== 컬럼 ===")
print(df.columns.tolist())
print(f"\n=== 재무제표 종류 분포 ===")
print(df["sj_nm"].value_counts())

전체 행 수: 215

=== 컬럼 ===
['rcept_no', 'reprt_code', 'bsns_year', 'corp_code', 'sj_div', 'sj_nm', 'account_id', 'account_nm', 'account_detail', 'thstrm_nm', 'thstrm_amount', 'frmtrm_nm', 'frmtrm_amount', 'ord', 'currency', 'thstrm_add_amount', 'frmtrm_q_nm', 'frmtrm_q_amount', 'frmtrm_add_amount']

=== 재무제표 종류 분포 ===
sj_nm
자본변동표      98
재무상태표      49
현금흐름표      38
손익계산서      17
포괄손익계산서    13
Name: count, dtype: int64


In [7]:
# 7개 원천 데이터 검색
targets = {
    "매출액": ["ifrs-full_Revenue", "ifrs_Revenue"],
    "영업이익": ["dart_OperatingIncomeLoss"],
    "당기순이익": ["ifrs-full_ProfitLoss"],
    "자본총계": ["ifrs-full_Equity"],
    "부채총계": ["ifrs-full_Liabilities"],
    "현금성자산": ["ifrs-full_CashAndCashEquivalents"],
    "감가상각비": ["dart_DepreciationExpense", "dart_DepreciationAndAmortisationExpense"],
}

print("=" * 70)
for name, account_ids in targets.items():
    found = df[df["account_id"].isin(account_ids)]
    if not found.empty:
        for _, row in found.iterrows():
            amount = row["thstrm_amount"]
            sj = row["sj_nm"]
            print(f"✅ {name:8s} | {sj:15s} | {row['account_id']:50s} | {amount}")
    else:
        print(f"❌ {name:8s} | 계정 ID 없음 → 다른 ID 탐색 필요")
print("=" * 70)

✅ 매출액      | 손익계산서           | ifrs-full_Revenue                                  | 133873444000000
✅ 영업이익     | 손익계산서           | dart_OperatingIncomeLoss                           | 57232797000000
✅ 당기순이익    | 손익계산서           | ifrs-full_ProfitLoss                               | 47225272000000
✅ 당기순이익    | 포괄손익계산서         | ifrs-full_ProfitLoss                               | 47225272000000
✅ 당기순이익    | 현금흐름표           | ifrs-full_ProfitLoss                               | 47225272000000
✅ 당기순이익    | 자본변동표           | ifrs-full_ProfitLoss                               | 0
✅ 당기순이익    | 자본변동표           | ifrs-full_ProfitLoss                               | 47101190000000
✅ 당기순이익    | 자본변동표           | ifrs-full_ProfitLoss                               | 0
✅ 당기순이익    | 자본변동표           | ifrs-full_ProfitLoss                               | 47101190000000
✅ 당기순이익    | 자본변동표           | ifrs-full_ProfitLoss                               | 124082000000
✅ 당기순이익    | 자본변동표           | ifrs-f

In [8]:
# 현금흐름표만 추출
cf = df[df["sj_nm"] == "현금흐름표"]
print(f"현금흐름표 항목 수: {len(cf)}")

# 감가상각 관련 키워드로 검색
print("\n=== '감가' 포함 항목 ===")
depreciation = cf[cf["account_nm"].str.contains("감가|상각|Depreciation|Amortis", na=False, case=False)]
print(depreciation[["account_id", "account_nm", "thstrm_amount"]].to_string())

현금흐름표 항목 수: 38

=== '감가' 포함 항목 ===
Empty DataFrame
Columns: [account_id, account_nm, thstrm_amount]
Index: []


In [9]:
def get_financial_data(corp_code: str, year: int, reprt_code: str, fs_div: str = "CFS") -> dict:
    """
    DART에서 단일 회사의 재무제표를 가져와 7개 원천 데이터를 추출한다.
    
    Args:
        corp_code: DART 기업 고유번호 (8자리)
        year: 사업연도 (예: 2024)
        reprt_code: 보고서 코드
            - 11011: 사업보고서 (4분기, 연간)
            - 11014: 3분기보고서
            - 11012: 반기보고서 (2분기)
            - 11013: 1분기보고서
        fs_div: CFS(연결) 또는 OFS(별도)
    
    Returns:
        dict: {매출액, 영업이익, 당기순이익, 감가상각비, 자본총계, 부채총계, 현금성자산}
              각 값은 정수(원 단위) 또는 None (데이터 없음)
    """
    url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
    params = {
        "crtfc_key": api_key,
        "corp_code": corp_code,
        "bsns_year": str(year),
        "reprt_code": reprt_code,
        "fs_div": fs_div,
    }
    
    response = requests.get(url, params=params, timeout=10)
    data = response.json()
    
    # 응답 검증
    if data.get("status") != "000":
        return {
            "error": f"DART 응답 오류: {data.get('status')} - {data.get('message')}",
            "corp_code": corp_code,
            "year": year,
            "reprt_code": reprt_code,
        }
    
    df = pd.DataFrame(data["list"])
    
    # 계정 ID 매핑 (회사마다 다를 수 있어 후보 여러 개)
    target_ids = {
        "매출액": ["ifrs-full_Revenue", "ifrs_Revenue"],
        "영업이익": ["dart_OperatingIncomeLoss"],
        "당기순이익": ["ifrs-full_ProfitLoss"],
        "자본총계": ["ifrs-full_Equity"],
        "부채총계": ["ifrs-full_Liabilities"],
        "현금성자산": ["ifrs-full_CashAndCashEquivalents"],
        "감가상각비": [
            "dart_DepreciationAndAmortisationExpense",
            "dart_DepreciationExpense",
        ],
    }
    
    result = {
        "corp_code": corp_code,
        "year": year,
        "reprt_code": reprt_code,
        "fs_div": fs_div,
    }
    
    for name, ids in target_ids.items():
        found = df[df["account_id"].isin(ids)]
        if not found.empty:
            # thstrm_amount는 문자열 (콤마 포함) → 정수 변환
            amount_str = found.iloc[0]["thstrm_amount"]
            try:
                # 콤마 제거 + 빈 문자열 처리
                amount = int(amount_str.replace(",", "")) if amount_str else None
            except (ValueError, AttributeError):
                amount = None
            result[name] = amount
        else:
            result[name] = None
    
    return result


# 함수 테스트 — 삼성전자 2024년 사업보고서
samsung_2024 = get_financial_data(
    corp_code="00126380",
    year=2024,
    reprt_code="11011",
    fs_div="CFS",
)

print("=== 삼성전자 2024년 (연결) ===")
for k, v in samsung_2024.items():
    if isinstance(v, int):
        print(f"  {k:12s}: {v:>20,} 원")
    else:
        print(f"  {k:12s}: {v}")

=== 삼성전자 2024년 (연결) ===
  corp_code   : 00126380
  year        :                2,024 원
  reprt_code  : 11011
  fs_div      : CFS
  매출액         :  300,870,903,000,000 원
  영업이익        :   32,725,961,000,000 원
  당기순이익       :   34,451,351,000,000 원
  자본총계        :  402,192,070,000,000 원
  부채총계        :  112,339,878,000,000 원
  현금성자산       :   53,705,579,000,000 원
  감가상각비       : None


In [10]:
# 최근 4분기 데이터 수집
# 2024년: 1Q, 2Q, 3Q, 사업보고서(연간) = 4개 보고서
# 분기별 데이터를 모두 가져온다

quarters = [
    ("2024", "11013", "2024_1Q"),  # 2024 1분기
    ("2024", "11012", "2024_2Q"),  # 2024 반기 (1~2분기 누적)
    ("2024", "11014", "2024_3Q"),  # 2024 3분기 (1~3분기 누적)
    ("2024", "11011", "2024_FY"),  # 2024 사업보고서 (연간)
]

print("4개 보고서 수집 중...\n")
results = []
for year, reprt_code, label in quarters:
    print(f"[{label}] 수집 중...", end=" ")
    data = get_financial_data("00126380", int(year), reprt_code)
    if "error" in data:
        print(f"❌ {data['error']}")
    else:
        print("✅")
    data["label"] = label
    results.append(data)

# DataFrame으로 정리
df_results = pd.DataFrame(results)
display_cols = ["label", "매출액", "영업이익", "당기순이익", "자본총계", "감가상각비"]
print(f"\n=== 분기별 데이터 ===")
print(df_results[display_cols].to_string())

4개 보고서 수집 중...

[2024_1Q] 수집 중... ✅
[2024_2Q] 수집 중... ✅
[2024_3Q] 수집 중... ✅
[2024_FY] 수집 중... ✅

=== 분기별 데이터 ===
     label              매출액            영업이익           당기순이익             자본총계 감가상각비
0  2024_1Q   71915601000000   6606009000000   6754708000000  371916124000000  None
1  2024_2Q   74068302000000  10443878000000   9841345000000  383526671000000  None
2  2024_3Q   79098731000000   9183371000000  10100904000000  386281363000000  None
3  2024_FY  300870903000000  32725961000000  34451351000000  402192070000000  None


In [11]:
# 검증: 우리가 가져온 값이 누적인지 단독인지 확인
# 단서: 손익계산서 항목의 frmtrm_nm, frmtrm_amount (전기 비교)

# 2024 3분기 보고서를 다시 직접 호출해서 원본 응답 확인
url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
params = {
    "crtfc_key": api_key,
    "corp_code": "00126380",
    "bsns_year": "2024",
    "reprt_code": "11014",  # 3분기보고서
    "fs_div": "CFS",
}

response = requests.get(url, params=params)
data = response.json()
df_raw = pd.DataFrame(data["list"])

# 매출액 행만 보기
revenue_rows = df_raw[df_raw["account_id"] == "ifrs-full_Revenue"]
print("=== 3분기보고서의 매출액 관련 행 ===")
print(revenue_rows[["sj_nm", "account_nm", "thstrm_nm", "thstrm_amount", "frmtrm_nm", "frmtrm_amount"]].to_string())

=== 3분기보고서의 매출액 관련 행 ===
    sj_nm account_nm   thstrm_nm   thstrm_amount frmtrm_nm frmtrm_amount
67  손익계산서        매출액  제 56 기 3분기  79098731000000       NaN           NaN


In [13]:
def get_financial_data_final(
    corp_code: str,
    year: int,
    reprt_code: str,
    fs_div: str = "CFS",
) -> dict:
    """
    최종 버전: 7개 원천 데이터 추출 + 업종 특성 감지
    """
    url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
    params = {
        "crtfc_key": api_key,
        "corp_code": corp_code,
        "bsns_year": str(year),
        "reprt_code": reprt_code,
        "fs_div": fs_div,
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as e:
        return {"error": f"네트워크 오류: {e}", "corp_code": corp_code}
    
    if data.get("status") != "000":
        return {
            "error": f"DART {data.get('status')}: {data.get('message')}",
            "corp_code": corp_code,
        }
    
    df = pd.DataFrame(data["list"])
    
    target_ids = {
        "매출액": ["ifrs-full_Revenue", "ifrs_Revenue"],
        "영업이익": ["dart_OperatingIncomeLoss"],
        "당기순이익": ["ifrs-full_ProfitLoss"],
        "자본총계": ["ifrs-full_Equity"],
        "부채총계": ["ifrs-full_Liabilities"],
        "현금성자산": ["ifrs-full_CashAndCashEquivalents"],
        "감가상각비": [
            "dart_DepreciationAndAmortisationExpense",
            "dart_DepreciationExpense",
        ],
    }
    
    result = {
        "corp_code": corp_code,
        "year": year,
        "reprt_code": reprt_code,
        "fs_div": fs_div,
    }
    
    for name, ids in target_ids.items():
        found = df[df["account_id"].isin(ids)]
        if not found.empty:
            amount_str = found.iloc[0]["thstrm_amount"]
            try:
                amount = int(amount_str.replace(",", "")) if amount_str and amount_str.strip() else None
            except (ValueError, AttributeError):
                amount = None
            result[name] = amount
        else:
            result[name] = None
    
    # 업종 특성 감지
    if result["매출액"] is None and result["영업이익"] is None:
        result["업종특성"] = "금융업 추정 (매출/영업이익 표준 항목 없음)"
        result["v1_지원"] = False
    elif result["매출액"] is None or result["당기순이익"] is None:
        result["업종특성"] = "비표준 회계 (일부 데이터 누락)"
        result["v1_지원"] = False
    else:
        result["업종특성"] = "일반 기업"
        result["v1_지원"] = True
    
    return result


# 다시 5종목 테스트
print("=" * 80)
print("Day 3 최종 검증 - 업종 특성 자동 감지")
print("=" * 80)

results = []
for name, corp_code in test_companies.items():
    data = get_financial_data_final(corp_code, 2024, "11014")
    data["회사명"] = name
    results.append(data)

df_final = pd.DataFrame(results)
print(df_final[["회사명", "업종특성", "v1_지원", "매출액", "당기순이익"]].to_string(index=False))

Day 3 최종 검증 - 업종 특성 자동 감지
   회사명                      업종특성  v1_지원          매출액          당기순이익
  삼성전자                     일반 기업   True 7.909873e+13 10100904000000
SK하이닉스                     일반 기업   True 1.757307e+13  5753373000000
   카카오                     일반 기업   True 1.921398e+12    78513044412
  셀트리온                     일반 기업   True 8.819333e+11    84094048278
  KB금융 금융업 추정 (매출/영업이익 표준 항목 없음)  False          NaN  1596039000000
